# Tidy Models

In [5]:
#import packages
library(tidymodels)
library(tidyverse)

In [3]:
#set options
set.seed(123)

In [6]:
#import data
nhanes <- read_csv("../data/nhanes.csv")

Rows: 20293 Columns: 78
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (29): SurveyYr, Gender, Race1, Race3, Education, MaritalStatus, HHIncome, HomeOwn, Work, BMICatUnder20yrs, BMI...
dbl (49): ID, Age, AgeMonths, HHIncomeMid, Poverty, HomeRooms, Weight, Length, HeadCirc, Height, BMI, Pulse, BPSys...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [11]:
nhanes |> 
  select(Height, Age, Sex=Gender, Race=Race1, Weight) |> 
  na.omit() |> #remove rows with missing
  sample_n(100) -> random.kids

In [12]:
random.kids

# A tibble: 100 × 5
   Height   Age Sex    Race    Weight
    <dbl> <dbl> <chr>  <chr>    <dbl>
 1  167.     14 male   White     58.6
 2  175.     46 male   Black     93.2
 3  165.     75 female Black     95.2
 4  137.     10 female Black     30.2
 5  152.     42 female Mexican   77.6
 6  169.     14 female Black     51.1
 7  165.     48 male   Other     82.4
 8  105.      3 male   Black     17  
 9   96.8     2 female Black     15.8
10  154.     56 female White     95.8
# ℹ 90 more rows
# ℹ Use `print(n = ...)` to see more rows

#### Paradigm for tidymodels

```mermaid
flowchart LR 

A(Begin ML) --> B[(Wrangle)] --> C[/Select model/] --> D[Evaluate model] --> E(Save model)

```

```mermaid
flowchart LR 

C[Create recipe] --> D[/"Build model"/] --> E[Create a workflow] --> F[Evaluate model]

```

In [14]:
#Preprocessing and recipe
recipe(Height ~ Age + Sex + Race + Weight, random.kids) -> kids.recipe

In [15]:
kids.recipe


── Recipe ────────────────────────────────────────────────────────────────────────────────────────────────────────────

── Inputs 
Number of variables by role
outcome:   1
predictor: 4

In [16]:
#model building
linear_reg() |> set_mode("regression") |> set_engine("lm") -> kids.model

In [19]:
kids.model

Linear Regression Model Specification (regression)

Computational engine: lm 


In [24]:
#workflow
workflow() |> add_recipe(kids.recipe) |> add_model(kids.model) -> kids.workflow

In [26]:
#fit model
kids.workflow |> fit(random.kids) -> kids.fitted.model

In [30]:
kids.fitted.model |> extract_fit_engine() |> tidy()

# A tibble: 8 × 5
  term         estimate std.error statistic  p.value
  <chr>           <dbl>     <dbl>     <dbl>    <dbl>
1 (Intercept)   109.       3.30      33.0   9.89e-53
2 Age             0.100    0.0673     1.49  1.40e- 1
3 Sexmale         7.64     2.53       3.01  3.32e- 3
4 RaceHispanic    4.81     4.06       1.18  2.40e- 1
5 RaceMexican    -1.03     3.92      -0.263 7.93e- 1
6 RaceOther       3.23     4.39       0.735 4.64e- 1
7 RaceWhite       5.15     3.50       1.47  1.44e- 1
8 Weight          0.535    0.0523    10.2   7.81e-17

In [31]:
kids.fitted.model |> extract_fit_engine() |> glance()

# A tibble: 1 × 12
  r.squared adj.r.squared sigma statistic  p.value    df logLik   AIC   BIC deviance df.residual  nobs
      <dbl>         <dbl> <dbl>     <dbl>    <dbl> <dbl>  <dbl> <dbl> <dbl>    <dbl>       <int> <int>
1     0.766         0.748  11.7      42.9 2.61e-26     7  -384.  786.  809.   12618.          92   100

In [35]:
kids.fitted.model |> predict(random.kids) -> kids.height.pred

In [44]:
kids.height.pred |> with(.pred)

  [1] 154.3899 170.9452 167.2805 126.0101 153.5324 137.5878 168.6001 125.8880 117.5079 170.8517 168.1199 156.4888 116.5248 178.2737 129.2368 148.6046 156.1813
 [18] 158.9405 182.1993 161.7797 148.4469 144.4919 134.1651 174.3560 168.2320 173.7307 126.6376 128.7077 174.9193 164.2919 152.3663 195.9617 164.9485 166.3922
 [35] 170.6671 119.2455 130.8560 142.6383 165.9204 188.0471 154.2593 187.2554 149.2428 145.1273 153.4359 138.8950 138.0199 196.9725 125.4067 162.7729 190.7128
 [52] 145.1623 174.7634 114.1235 116.8908 151.8135 159.0813 132.2792 163.1097 150.0894 125.9959 147.5349 173.5865 131.7366 166.3222 175.6339 174.1049 155.9289
 [69] 143.5955 156.2145 122.5009 136.6279 158.0133 154.9510 169.6282 148.6277 169.1276 127.7636 131.3225 184.7652 129.3893 142.3085 154.9968 173.5538 139.3045
 [86] 135.6118 168.2367 179.7524 124.9400 134.9312 120.1296 128.1765 165.9680 188.5748 133.1624 166.8191 151.2981 166.1537 154.3105 139.9216

In [45]:
random.kids$pred_height = kids.height.pred |> with(.pred)

In [46]:
random.kids

# A tibble: 100 × 6
   Height   Age Sex    Race    Weight pred_height
    <dbl> <dbl> <chr>  <chr>    <dbl>       <dbl>
 1  167.     14 male   White     58.6        154.
 2  175.     46 male   Black     93.2        171.
 3  165.     75 female Black     95.2        167.
 4  137.     10 female Black     30.2        126.
 5  152.     42 female Mexican   77.6        154.
 6  169.     14 female Black     51.1        138.
 7  165.     48 male   Other     82.4        169.
 8  105.      3 male   Black     17          126.
 9   96.8     2 female Black     15.8        118.
10  154.     56 female White     95.8        171.
# ℹ 90 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
#evaluation
library(yardstick)

In [34]:
metrics <- metric_set(rmse, mape)

In [48]:
random.kids |> metrics(Height, pred_height)

# A tibble: 2 × 3
  .metric .estimator .estimate
  <chr>   <chr>          <dbl>
1 rmse    standard       11.2 
2 mape    standard        6.29

In [49]:
ls()

[1] "kids.fitted.model" "kids.height.pred"  "kids.model"        "kids.recipe"       "kids.workflow"     "metrics"           "nhanes"            "random.kids"      

In [51]:
save(kids.fitted.model, file='fitted_model_lm_kids.rdata')

In [ ]:
recipe(Height ~ Age + Gender + Race1 + Weight, nhanes) |> 
  step_naomit(everything()) |> 
  step_dummy(all_factor_predictors()) |> 
  step_normalize(all_numeric_predictors()) 

# A tibble: 20,293 × 8
        Age Weight Height Gender_male Race1_Hispanic Race1_Mexican Race1_Other Race1_White
      <dbl>  <dbl>  <dbl>       <dbl>          <dbl>         <dbl>       <dbl>       <dbl>
 1 -0.00874  0.706   165.       1.00          -0.347        -0.464      -0.360       1.31 
 2 -1.27    -1.67    105.       1.00          -0.347        -0.464       2.78       -0.766
 3 -0.763    0.197   181.       1.00          -0.347        -0.464      -0.360      -0.766
 4 -1.01    -0.899   148.       1.00          -0.347        -0.464      -0.360      -0.766
 5  1.08     1.70    166       -0.997         -0.347        -0.464      -0.360      -0.766
 6 -0.344    1.05    173        1.00          -0.347         2.15       -0.360      -0.766
 7  0.620    0.682   168.      -0.997         -0.347        -0.464      -0.360       1.31 
 8 -1.39    -1.92     NA       -0.997         -0.347        -0.464      -0.360       1.31 
 9 -1.01    -1.36    140.       1.00           2.88         -0.464 